In [81]:
import os
import sys
sys.path.append('../../MARS')
#sys.path.append('../../MARS/preprocess_dicoms')
#sys.path.append('../../MARS/run_lars')
import ast
import pandas as pd
import numpy as np
import os
import h5py
import pydicom 
import glob
#import tkinter
#os.environ['MPLCONFIGDIR'] = os.getcwd() + "/configs/"
import matplotlib
#matplotlib.use('TkAgg')
#matplotlib.use('qtagg') 
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import ImageGrid
#import itertools
import pickle
import scipy
import pdb
import time
import argparse
from collections import Counter
from MARS.preprocess_dicoms.read_dicom_functions import GetDicomSlices, GetImageFromDicomSlices, GetImageFromDicomSlices_PixelData, ImageBQML2SUV



In [4]:
log_files = glob.glob("/gpfs/data/mayerhoeferlab/projects/myeloma/process_dicoms_for_mars/log_for_*_*.log")   # adjust path if logs are in a subfolder

for op in log_files:
    with open(op, "r") as infile:
        # print the total number of lines in infile
        print(f"Total lines in {op}: {sum(1 for line in infile)}")

Total lines in /gpfs/data/mayerhoeferlab/projects/myeloma/process_dicoms_for_mars/log_for_6178131_45.log: 967
Total lines in /gpfs/data/mayerhoeferlab/projects/myeloma/process_dicoms_for_mars/log_for_6173745_49.log: 1029
Total lines in /gpfs/data/mayerhoeferlab/projects/myeloma/process_dicoms_for_mars/log_for_6177992_30.log: 1090
Total lines in /gpfs/data/mayerhoeferlab/projects/myeloma/process_dicoms_for_mars/log_for_6178133_47.log: 867
Total lines in /gpfs/data/mayerhoeferlab/projects/myeloma/process_dicoms_for_mars/log_for_6177995_33.log: 1500
Total lines in /gpfs/data/mayerhoeferlab/projects/myeloma/process_dicoms_for_mars/log_for_6176266_3.log: 1086
Total lines in /gpfs/data/mayerhoeferlab/projects/myeloma/process_dicoms_for_mars/log_for_6177993_31.log: 1150
Total lines in /gpfs/data/mayerhoeferlab/projects/myeloma/process_dicoms_for_mars/log_for_6176849_5.log: 1251
Total lines in /gpfs/data/mayerhoeferlab/projects/myeloma/process_dicoms_for_mars/log_for_6177998_36.log: 1117
Total

In [12]:


output_file = "/gpfs/data/mayerhoeferlab/projects/myeloma/process_dicoms_for_mars/merged_logs.log"

with open(output_file, "w") as outfile:
    for fname in log_files:
        with open(fname, "r") as infile:
            # Write filename as a header (optional)
            outfile.write(f"===== {fname} =====\n")
            outfile.write(infile.read())
            outfile.write("\n\n")  # spacer between logs

print(f"Merged {len(log_files)} logs into {output_file}")

Merged 50 logs into /gpfs/data/mayerhoeferlab/projects/myeloma/process_dicoms_for_mars/merged_logs.log


In [5]:
listo = glob.glob('/gpfs/data/mayerhoeferlab/projects/myeloma/process_dicoms_for_mars/datafiles/data_scan_*_0.csv')
list_df = []

for x in listo:
    dfo = pd.read_csv(x)
    list_df.append(dfo)

In [4]:
zz = pd.read_csv('/gpfs/data/mayerhoeferlab/projects/myeloma/process_dicoms_for_mars/datafiles/data_count_axial_pixel_size.csv')

/tmp/ipykernel_304775/678610853.py:1: DtypeWarning: Columns (59) have mixed types. Specify dtype option on import or set low_memory=False.
  zz = pd.read_csv('/gpfs/data/mayerhoeferlab/projects/myeloma/process_dicoms_for_mars/datafiles/data_count_axial_pixel_size.csv')


In [47]:
comb = pd.read_csv('/gpfs/data/mayerhoeferlab/projects/myeloma/process_dicoms_for_mars/datafiles/data_scan_curated.csv')
#comb['fullfile'] = comb['filepath'].astype(str) + comb['fullfile']

In [49]:
comb_all = pd.read_csv('/gpfs/data/mayerhoeferlab/projects/myeloma/process_dicoms_for_mars/datafiles/data_scan.csv')
comb_cur = pd.read_csv('/gpfs/data/mayerhoeferlab/projects/myeloma/process_dicoms_for_mars/datafiles/data_scan_curated.csv')

des_all = pd.read_csv('/gpfs/data/mayerhoeferlab/projects/myeloma/datafiles_for_mars/data_scan.csv')
des_cur = pd.read_csv('/gpfs/data/mayerhoeferlab/projects/myeloma/datafiles_for_mars/data_scan_curated.csv')

/tmp/ipykernel_400450/3800651476.py:1: DtypeWarning: Columns (59) have mixed types. Specify dtype option on import or set low_memory=False.
  comb_all = pd.read_csv('/gpfs/data/mayerhoeferlab/projects/myeloma/process_dicoms_for_mars/datafiles/data_scan.csv')
/tmp/ipykernel_400450/3800651476.py:4: DtypeWarning: Columns (59) have mixed types. Specify dtype option on import or set low_memory=False.
  des_all = pd.read_csv('/gpfs/data/mayerhoeferlab/projects/myeloma/datafiles_for_mars/data_scan.csv')


In [50]:
comb_all[comb_all['filepath'] == 12194426]

,filepath,filename,filename_header,fullfile,patient_id,patient_age,patient_sex,patient_weight,smoking_status,accession,...,reconstruction,corrections,pixel_unit,frame_duration,tracer,tracer_injection_time,tracer_dose,tracer_halflife,image_filter,contrast_route
1319,12194426,12194426_ct_1.h5,12194426_ct_1.pkl,/12194426_ct_1.h5,1597595,074Y,M,76.203528,NaN,12194426,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1320,12194426,12194426_pt_1.h5,12194426_pt_1.pkl,/12194426_pt_1.h5,1597595,074Y,M,76.203528,NaN,12194426,...,OSEM2D 1i21s,NORM|DTIM|RADL|DECY,PROPCPS,150000.0,FDG,123200.0,5.764600e+08,6586.2,NaN,NaN
1321,12194426,12194426_ct_2.h5,12194426_ct_2.pkl,/12194426_ct_2.h5,1597595,074Y,M,76.203528,NaN,12194426,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1322,12194426,12194426_pt_0.h5,12194426_pt_0.pkl,/12194426_pt_0.h5,1597595,074Y,M,76.203528,NaN,12194426,...,OSEM2D 2i21s,NORM|DTIM|ATTN|SCAT|RADL|DECY,BQML,150000.0,FDG,123200.0,5.764600e+08,6586.2,NaN,NaN
1323,12194426,12194426_ct_0.h5,12194426_ct_0.pkl,/12194426_ct_0.h5,1597595,074Y,M,76.203528,NaN,12194426,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
des.columns

0             PET WB Corrected
1                 WB_PET AC 3D
2             PET WB Corrected
3             PET WB Corrected
4             PET WB Corrected
                 ...          
1514          PET WB Corrected
1515          PET WB Corrected
1516    PET WB _Corrected (AC)
1517        PET VERTEX TO TOES
1518          PET WB Corrected
Name: description_series, Length: 1519, dtype: object

In [82]:
patho = '/gpfs/data/mayerhoeferlab/projects/myeloma/dicom/39708174'
xyz = GetDicomSlices(patho)

In [91]:
ur = [u.SeriesDescription for u in xyz]
ur_m = [u.Modality for u in xyz]

In [92]:
ur_m

['PT',
 'PT',
 'CT',
 'PT',
 'PT',
 'CT',
 'PT',
 'PT',
 'CT',
 'PT',
 'PT',
 'CT',
 'PT',
 'PT',
 'CT',
 'PT',
 'PT',
 'CT',
 'PT',
 'PT',
 'CT',
 'PT',
 'PT',
 'PT',
 'CT',
 'PT',
 'PT',
 'PT',
 'CT',
 'PT',
 'PT',
 'PT',
 'CT',
 'PT',
 'PT',
 'PT',
 'CT',
 'PT',
 'PT',
 'PT',
 'CT',
 'PT',
 'PT',
 'PT',
 'CT',
 'PT',
 'PT',
 'PT',
 'CT',
 'PT',
 'PT',
 'PT',
 'CT',
 'PT',
 'PT',
 'PT',
 'CT',
 'PT',
 'PT',
 'PT',
 'CT',
 'PT',
 'PT',
 'PT',
 'CT',
 'PT',
 'PT',
 'PT',
 'CT',
 'PT',
 'PT',
 'PT',
 'CT',
 'PT',
 'PT',
 'PT',
 'CT',
 'PT',
 'PT',
 'PT',
 'CT',
 'PT',
 'PT',
 'PT',
 'CT',
 'PT',
 'PT',
 'PT',
 'CT',
 'PT',
 'PT',
 'PT',
 'CT',
 'PT',
 'PT',
 'PT',
 'CT',
 'PT',
 'PT',
 'PT',
 'CT',
 'PT',
 'PT',
 'PT',
 'CT',
 'PT',
 'PT',
 'PT',
 'CT',
 'PT',
 'PT',
 'PT',
 'CT',
 'PT',
 'PT',
 'PT',
 'CT',
 'PT',
 'PT',
 'PT',
 'CT',
 'PT',
 'PT',
 'PT',
 'CT',
 'PT',
 'PT',
 'PT',
 'CT',
 'PT',
 'PT',
 'PT',
 'CT',
 'PT',
 'PT',
 'PT',
 'CT',
 'PT',
 'PT',
 'PT',
 'CT',
 'PT',
 'PT',

In [ ]:
zr = pd.read_pickle('/gpfs/data/mayerhoeferlab/projects/myeloma/process_dicoms_for_mars/headers_pkl/39708174/39708174_pt_0.pkl')
for x in zr:
    print(x.SeriesDescription)

In [71]:
#comb_all[comb_all.filepath == 39708174][['filename', 'description_series']]
des_all[des_all.filepath == 39708174][['filename', 'description_series']]

,filename,description_series
17864,39708174_pt_3.h5,Reformatted
17865,39708174_pt_4.h5,Reformatted
17866,39708174_pt_1.h5,WB_PET NAC 3D
17867,39708174_ct_0.h5,CTAC 3.75 thick
17868,39708174_pt_0.h5,WB_PET AC 3D
17869,39708174_pt_2.h5,Reformatted


In [62]:
comb[comb.filepath == 39708174].description_series

1097    WB_PET AC 3D
Name: description_series, dtype: object

In [51]:
def row_diff(dfA, dfB, colname = 'filename'):
    # get values that are in dfA but not dfB
    ret = set(dfA[colname].values) - set(dfB[colname].values)
    print(len(ret))
    return ret
row_diff(comb_all, des_all)
row_diff(comb_cur, des_cur)

row_diff(des_all, comb_all)
row_diff(des_cur, comb_cur)

800
51
0
7


{'12194426_pt_0.h5',
 '30273728_pt_0.h5',
 '38797362_pt_1.h5',
 '39708174_pt_0.h5',
 '40026035_pt_0.h5',
 '41997412_pt_0.h5',
 '6510655_pt_0.h5'}

In [ ]:
asdf = pd.read_csv('/gpfs/data/mayerhoeferlab/projects/myeloma/process_dicoms_for_mars/tmp.csv')
for i in range(asdf.shape[0]):
    xx = np.array(ast.literal_eval(asdf['image_orientation'].values[i]))
    crossed = np.cross(xx[:3], xx[3:])
    if not np.all(crossed == np.array([0, 0, 1])):
        print(i)
        print(np.round(crossed, 2))
        print('==')

In [55]:
des[des['imaging_plane'] == 'coronal']

,filepath,filename,filename_header,fullfile,patient_id,patient_age,patient_sex,patient_weight,accession,study_uid,...,img_mean_nonzeros,img_std_nonzeros,img_sum_nonzeros,convolution_kernel,image_filter,xray_tube_current,kvp,image_comments,smoking_status,contrast_route
5,2938443,2938443_pt_2.h5,2938443_pt_2.pkl,2938443/2938443_pt_2.h5,7287510,079Y,F,46.308000,2938443,1.3.12.2.1107.5.1.4.44843.30000005072920385017...,...,0.996660,0.993341,1.580680e+06,X-Y-Z Gaussian FWHM=5.00,NaN,NaN,NaN,NaN,NaN,NaN
36,4026061,4026061_pt_2.h5,4026061_pt_2.pkl,4026061/4026061_pt_2.h5,1629898,058Y,F,65.830000,4026061,1.2.840.113696.323972.500.62352.2007111410553,...,0.818885,0.886471,3.590883e+06,X-Y-Z Gaussian FWHM=5.00,NaN,NaN,NaN,NaN,NO,NaN
865,6158063,6158063_ct_0.h5,6158063_ct_0.pkl,6158063/6158063_ct_0.h5,2132971,061Y,F,86.260000,6158063,1.3.12.2.1107.5.1.4.48416.4.0.1430976512481679,...,4.932646,55.323428,1.348684e+06,T20s,NaN,50.0,120.0,NaN,NaN,NaN
1007,6340889,6340889_pt_3.h5,6340889_pt_3.pkl,6340889/6340889_pt_3.h5,1838443,062Y,M,81.646637,6340889,1.2.840.113696.323972.500.2332113.20111216114056,...,0.948266,1.043459,6.269947e+05,XYZ Gauss3.00,NaN,NaN,NaN,AC CT WB,UNKNOWN,NaN
1069,6409349,6409349_pt_3.h5,6409349_pt_3.pkl,6409349/6409349_pt_3.h5,1494896,075Y,F,80.285860,6409349,1.2.840.113696.323972.500.2399179.20120113112336,...,0.907374,1.401781,6.432403e+05,XYZ Gauss3.00,NaN,NaN,NaN,AC CT WB,UNKNOWN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19384,43486598,43486598_pt_2.h5,43486598_pt_2.pkl,43486598/43486598_pt_2.h5,16765894,055Y,M,95.255000,43486598,1.2.840.114350.2.232.2.798268.2.880366082.1,...,0.815434,1.186641,7.663502e+05,XYZ Gauss5.00,NaN,NaN,NaN,AC CT WB EFOV iMAR,UNKNOWN,NaN
19408,43608032,43608032_ct_4.h5,43608032_ct_4.pkl,43608032/43608032_ct_4.h5,12734299,061Y,M,100.699000,43608032,1.2.840.114350.2.232.2.798268.2.875513226.1,...,-275.117529,419.174127,-8.443480e+09,"['Br38f', '3']",NaN,211.0,120.0,NaN,UNKNOWN,NaN
19409,43608032,43608032_pt_2.h5,43608032_pt_2.pkl,43608032/43608032_pt_2.h5,12734299,061Y,M,100.699000,43608032,1.2.840.114350.2.232.2.798268.2.875513226.1,...,0.857930,2.501895,8.680667e+05,XYZ Gauss5.00,NaN,NaN,NaN,AC CT WB EFOV iMAR,UNKNOWN,NaN
19411,43644283,43644283_pt_3.h5,43644283_pt_3.pkl,43644283/43644283_pt_3.h5,9114915,091Y,M,75.751000,43644283,1.2.840.114350.2.232.2.798268.2.883783980.1,...,0.869360,1.020701,2.035668e+06,XYZ Gauss5.00,NaN,NaN,NaN,AC CT WB 5.0 HD_FoV,UNKNOWN,NaN


In [57]:
for i in range(des.shape[0]):
    xx = np.array(ast.literal_eval(des['image_orientation'].values[i]))
    crossed = np.cross(xx[:3], xx[3:])
    if not np.all(crossed == np.array([0, 0, 1])):
        print(i)
        print(np.round(crossed, 2))
        print('==')

5
[-0.  1.  0.]
==
36
[-0.  1.  0.]
==
865
[-0.  1.  0.]
==
1007
[-0.  1.  0.]
==
1069
[-0.  1.  0.]
==
1082
[-0.  1.  0.]
==
1093
[-0.  1.  0.]
==
1098
[-0.  1.  0.]
==
1135
[-0.  1.  0.]
==
1157
[-0.  1.  0.]
==
1178
[-0.  1.  0.]
==
1190
[-0.  1.  0.]
==
1211
[-0.  1.  0.]
==
1231
[-0.  1.  0.]
==
1240
[-0.  1.  0.]
==
1246
[-0.  1.  0.]
==
1251
[-0.  1.  0.]
==
1260
[-0.  1.  0.]
==
1270
[-0.  1.  0.]
==
1282
[-0.  1.  0.]
==
1294
[-0.  1.  0.]
==
1302
[-0.  1.  0.]
==
1317
[-0.  1.  0.]
==
1322
[-0.  1.  0.]
==
1337
[-0.  1.  0.]
==
1350
[-0.  1.  0.]
==
1360
[-0.  1.  0.]
==
1367
[-0.  1.  0.]
==
1369
[-0.  1.  0.]
==
1381
[-0.  1.  0.]
==
1386
[-0.  1.  0.]
==
1389
[-0.  1.  0.]
==
1398
[-0.  1.  0.]
==
1409
[-0.  1.  0.]
==
1414
[-0.  1.  0.]
==
1420
[-0.  1.  0.]
==
1426
[-0.  1.  0.]
==
1436
[-0.  1.  0.]
==
1454
[-0.  1.  0.]
==
1469
[-0.  1.  0.]
==
1479
[-0.  1.  0.]
==
1482
[-0.  1.  0.]
==
1506
[-0.  1.  0.]
==
1512
[-0.  1.  0.]
==
1547
[-0.  1.  0.]
==
1571
[-0.  1.  0

In [ ]:
asdf = pd.read_pickle('/gpfs/data/mayerhoeferlab/projects/myeloma/process_dicoms_for_mars/headers_pkl/13224275/13224275_pt_0.pkl')

AttributeError: 'FileDataset' object has no attribute 'corrections'

In [23]:
diff = list(set(comb['filename'].values) - set(des['filename'].values))

In [26]:
comb[comb.filename.isin(diff)].to_csv('/gpfs/data/mayerhoeferlab/projects/myeloma/process_dicoms_for_mars/tmp.csv')